# S403011 Machine Learning Project: Weather Prediction

Team:  Power Rangers

Authors: Katrin Kulberg (25-327-875), Eric Lee (22-311-021), Rafael Pereira Tinoco (22-306-443)

# Install packages if needed

In [ ]:
!pip install numpy pandas scikit-learn optuna plotnine seaborn matplotlib xgboost optuna_integration

## Importing packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, RobustScaler, MaxAbsScaler, PowerTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import xgboost
from optuna.samplers import TPESampler
import optuna
import optuna_integration
from optuna.visualization import plot_optimization_history
from plotnine import ggplot, aes, geom_point, geom_line, facet_wrap, labs, theme_bw, theme, scale_color_manual, geom_histogram, geom_boxplot, scale_x_discrete
import seaborn as sns
import matplotlib.pyplot as plt
import json
#For the map
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## General Function Definitions

In [ ]:
#Function that, given an array of temperatures,
#pressures and relative humidities,
#solves the Arden Buck equation to calculate
#the partial water vapor pressure in the atmosphere
def solve_arden_buck(temps, pres, rh):
  e_coeffs = (6.1121, 18.729, 257.87, 227.3)
  f_coeffs = (4.1e-4, 3.48e-6, 7.4e-10, 30.6, -3.8e-2)
  
  this_e = e_coeffs[0] * np.exp((e_coeffs[1] - temps/e_coeffs[3]) * temps/(temps + e_coeffs[2]))
  this_f = 1 + f_coeffs[0] + pres * (f_coeffs[1] + f_coeffs[2] * np.square(temps + f_coeffs[3] + f_coeffs[4] * pres))
  
  vapor_pressure = this_e * this_f
  
  return vapor_pressure * rh

In [ ]:
all_stations = ("ANT", "BAS", "DAV", "DOL", "GVE", "INT", "LUG", "SIO", "STG", "ZER")
stations_for_pp0q = ("BAS", "GVE", "INT", "LUG", "SIO")

vars_for_normal    = ("prestah0_", "tre200h0_", "vaporpre_")
vars_for_maxabs    = ("gre000h0_", "rre150h0_", "sre000h0_", "fkl010h0_", "fkl010h3_")

columns_for_normal = ( [v + s for v in vars_for_normal for s in all_stations] + 
                       ["tre200h0", "tre200h0_lag24h"] +
                       ["pp0qffh0_" + s for s in stations_for_pp0q]              )
columns_for_maxabs = [v + s for v in vars_for_maxabs for s in all_stations]

#Function that fits the different scalers and transformers
#we will need given the training data.
def get_data_transformers(data_X):
  normal =   StandardScaler().fit(data_X[columns_for_normal])
  maxabs =     MaxAbsScaler().fit(data_X[columns_for_maxabs])
  return (normal, maxabs)

#Function that transforms data according to previously fitted
#scalers and transformers.
def transform_data(transformers, data_X):
  robust, maxabs = transformers
  transformed_data = data_X.copy()
  transformed_data[columns_for_normal ] = robust.transform(data_X[columns_for_normal])
  transformed_data[columns_for_maxabs ] = maxabs.transform(data_X[columns_for_maxabs])
  return transformed_data

In [ ]:
#Function that reads and pre-processes data from a file.
#Outputs the X and Y for training, or X and None for the Kaggle predictions.
def load_data(input_file):
  data = pd.read_csv(input_file)
  
  #Decompose the hour in sine and cosine
  #for better modelling of the daily temperature variation
  data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
  data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)
  
  #Remove missing values
  print("Before:", data.shape)
  data = data.dropna()
  print("After:", data.shape)
  
  #Encode season
  my_ordinal_encoder = OrdinalEncoder()
  my_ordinal_encoder = OrdinalEncoder(categories = [['Spring', 'Summer', 'Autumn', 'Winter']])
  data["season_enc"] = my_ordinal_encoder.fit_transform(data[["season"]])
  #Get sine and cosine of season
  #for better modelling of the yearly trends
  data["season_sin"] = np.sin(2 * np.pi * (data["season_enc"]) / 4)
  data["season_cos"] = np.cos(2 * np.pi * (data["season_enc"] )  / 4)

  #Calculate the partial vapor pressure
  #to avoid the significant correlation
  #of relative humidity and temperature
  #that might confuse the models...
  for s in all_stations:
    data["vaporpre_" + s] = solve_arden_buck(data["tre200h0_" + s], data["prestah0_" + s], data["ure200h0_" + s])

  #Check if all the expected targets are present in the input:
  #if not, assume this is test data...
  target_columns = ["target_tre200h0_plus12h", "target_tre200h0_plus24h", "target_tre200h0_plus48h"]
  is_training = True
  for c in target_columns:
    if c not in data.columns:
      is_training = False
      break
  
  #Drop the data we don't want to use for training
  #(errors = 'ignore' to suppress the case
  # where any or all of the targets are not present
  # and also to cover the Id for the Kaggle competition input)
  X = data.drop(["target_tre200h0_plus12h",
                 "target_tre200h0_plus24h", 
                 "target_tre200h0_plus48h", 
                 "hour", 
                 "season", 
                 "season_enc",
                 "Id"] + ["ure200h0_{}".format(s) for s in all_stations],
                axis=1, errors='ignore')
  
  if is_training:
    Y = data[target_columns]
  else:
    Y = None
  
  return (X, Y, data["season_enc"])

In [ ]:
#Set up some (global) parameters.
#The random seeds ensure determinism.
default_tuning_folds_number = 7
default_tuning_folds_random_seed = 42
default_tuning_sampler_random_seed = 42
default_tuning_shuffler_random_seed = 42
default_tuning_folds_before_pruning = 2
default_tuning_n_startup_trials = 10
default_tuning_p_threshold = 0.1
default_tuning_n_jobs = -1
default_tuning_n_startup_trials = 10

#Function that receives a callable that returns a model
#given a set of parameters, a callable that returns
#the training parameters given the Optuna trial,
#and the (scaled) training data. If either of
#training_input or training_output are callable,
#they receives instead the column (i. e. 12, 24 or 48),
#fold index and training indices to use and should
#return the relevant data points to be used.
#Returns a list containing the tuples of (best_mae, best_params)
#for all the targets.
#Optional parameters are:
#an iterable containing the target
#durations (if only a subset of 12, 24 and 48 is desired);
#a function to split the input data into folds
#(takes input data as parameters, returns an array
# of pairs of arrays containing the indices of the
# k-1 training folds and the k folds: [(train_1, val_1), (train_2, val_2)...]
# where all the train_i and val_i are vectors of indices);
#a function that receives the model, X and Y and fits the former to the latter two;
#a function that receives the column, fold index and validation indices
#and returns the corresponding input values to be used for validating the model
#(this is to support the sophisticated model where we fit to OOF residuals
# and validate with the residual from the overall fitted model);
#a function that receives the column, fold index and validation indices
#and returns the corresponding output values to be used for validating the model
#(this is to support the sophisticated model where we fit to OOF residuals
# and validate with the residual from the overall fitted model);
#an array of seasons (0, 1, 2 or 3), which, if not None,
#will be used to fit different models per season;
#the total number of trials to tune (defaulting to 100);
#whether to show the optimisation plot for each target (defaulting to False);
#the total number of jobs to use (defaulting to -1 for using all cores);
#a way to override the whole CV folding procedure that, if not none,
#receives the Optuna trial, the parameters, the model, the folds and the data
#and is expected to return the MAE, requiring the input and output to just be
#provided as DataFrames or ndarrays (so no callables or different validation, etc.).
def tune_model(create_model, get_params, training_input, training_output,
               targets = (12, 24, 48),
               split_folds = None,
               fit_model = None,
               validation_input = None,
               validation_output = None,
               seasons = None,
               num_total_trials = 100,
               show_optimisation_plot = False,
               tuning_n_jobs = default_tuning_n_jobs,
               override_CV = None):
  
  ret = []
  
  #By default, we use default_tuning_folds_number folds.
  #We override this for tuning the two-model approach,
  #both to have more folds and to guarantee we use the correct
  #precalculated values for the forests.
  if split_folds is None:
    cross_validation_fold_splitter = KFold(n_splits     = default_tuning_folds_number,
                                           random_state = default_tuning_folds_random_seed,
                                           shuffle = True)
    fold_indices = [(train, val) for train, val in cross_validation_fold_splitter.split(training_input)]
  else:  
    fold_indices = split_folds(training_input)
  
  #By default, we simply fit,
  #but we may need to override this
  #for more sophisticated models
  real_fitting_function = (lambda m, x, y: m.fit(x, y)) if fit_model is None else fit_model

  #If the training_inputt is callable,
  #we assume it takes column, index and training indices
  #to return the actual data; if not, it is the data...
  training_input_getter = (lambda column, idx, t_idx: training_input.iloc[t_idx]) if not callable(training_input) else training_input
  #By default, we use the same input for validation too,
  #but the more sophisticated model will require a different approach
  validation_input_getter = training_input_getter if validation_input is None else validation_input

  #If the training_output is callable,
  #we assume it takes column, index and training indices
  #to return the actual data; if not, it is the data...
  training_target_getter = (lambda column, idx, t_idx: training_output[column].iloc[t_idx]) if not callable(training_output) else training_output
  #By default, we use the same output for validation too,
  #but the more sophisticated model will require a different approach
  validation_target_getter = training_target_getter if validation_output is None else validation_output

  if seasons is not None:
    unique_season = seasons.unique()
    season_locator_array = [np.array(seasons == t) for t in unique_season]

  for target_t in targets:
    target_column = "target_tre200h0_plus{}h".format(target_t)

    generator = np.random.default_rng(default_tuning_shuffler_random_seed)
    
    def optuna_objective_function(trial):
      params = get_params(trial)
      model = create_model(params)

      if override_CV is not None:
        return override_CV(trial, params, model, fold_indices, training_input, training_output[target_column])
      else:
        mae = 0.0
        
        shuffled_indices = generator.permutation(len(fold_indices))
        
        for total_is_so_far, i in enumerate(shuffled_indices):
          train_idxs, val_idxs = fold_indices[i]
          
          to_train_X = training_input_getter(target_column, i, train_idxs)
          to_train_Y = training_target_getter(target_column, i, train_idxs)
          
          to_test_X = validation_input_getter(target_column, i, val_idxs)
          to_test_Y = validation_target_getter(target_column, i, val_idxs)
          
          if seasons is not None:
            this_mae = 0
            for locator in season_locator_array:
              real_fitting_function(model, to_train_X[locator[train_idxs]], to_train_Y[locator[train_idxs]])
              this_mae += mean_absolute_error(to_test_Y[locator[val_idxs]], model.predict(to_test_X[locator[val_idxs]]))
            this_mae /= len(unique_season)
          else:
            real_fitting_function(model, to_train_X, to_train_Y)
            this_mae = mean_absolute_error(to_test_Y, model.predict(to_test_X))
          
          trial.report(this_mae, i)
          
          mae += this_mae
          if trial.should_prune():
            return mae / (total_is_so_far + 1)
        
        return mae / len(fold_indices)
    
    
    study = optuna.create_study(
      direction = 'minimize',
      sampler = TPESampler(seed = default_tuning_sampler_random_seed, n_startup_trials = default_tuning_n_startup_trials),
      pruner = optuna.pruners.WilcoxonPruner(n_startup_steps = default_tuning_folds_before_pruning)
    )
    
    print("Tuning for: ", target_t)
    
    study.optimize(optuna_objective_function, n_trials = num_total_trials, n_jobs = tuning_n_jobs)
    
    num_pruned_trials = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    num_complete_trials = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)

    print(f"  Study statistics for {target_t}h: ")
    print(f"    Number of finished trials: {len(study.trials)}")
    print(f"    Number of pruned trials: {num_pruned_trials}")
    print(f"    Number of complete trials: {num_complete_trials}")
    
    if show_optimisation_plot:
      plot_optimization_history(study).show()
    
    ret += [(study.best_value, study.best_params | study.best_trial.user_attrs)]
  
  return ret

In [ ]:
#Function that assesses a model.
#Takes as arguments the a callable that,
#given parameters, returns the model,
#an iterable with the best parameters for every target,
#X and Y for training, X and Y for validation,
#(full) X and Y for training the full model for the Kaggle competition,
#and X for the Kaggle competition.
#Returns a list containing, for every target,
#a tuple containing (prediction, residual, MAE) for training and validation;
#for the 24h target, this will also contain predictions for Kaggle.
#In short: ((train_pred, train_res, train_MAE), (val_pred, val_res, val_MAE), Kaggle_pred).
#Optional parameters are:
#an iterable containing the target
#durations (if only a subset of 12, 24 and 48 is desired);
#a function that receives the model, X and Y and fits the former to the latter two;
#a function that receives the model and X and returns an array of predictions;
#a boolean that can skip the predictions for Kaggle if they are not desired (defaults to False -> not skipping);
#a boolean that controls whether to pass current target
#and what kind of operation (0 -> train fit, 1 -> train pred,
#2 -> test pred, 3 -> Kaggle train, 4 -> Kaggle pred)
#to aid in caching;
#a list of 4 season arrays that (if not None) will be passed
#to training, validation, competition training and competition predictions
#respectively if these use a custom function (after the previous if caching is true);
def assess_model(create_model, best_params,
                 training_input, training_output,
                 validation_input, validation_output,
                 competition_train_input, competition_train_output,
                 competition_pred_input,
                 targets = (12, 24, 48),
                 fit_model = None,
                 predict_with_model = None,
                 skip_competition_predictions = False,
                 caching_helper = False,
                 seasons = None):
  ret = []
  
  real_fit_function = (lambda m, x, y, s = None, t = None, i = None: m.fit(x, y)) if fit_model is None else fit_model
  real_pred_function = (lambda m, x, s = None, t = None, i = None: m.predict(x)) if predict_with_model is None else predict_with_model
  
  for target_t, params in zip(targets, best_params):
    target_column = "target_tre200h0_plus{}h".format(target_t)
    
    model = create_model(params)

    real_fit_function(*(t for t in (model,
                                    training_input,
                                    training_output[target_column],
                                    target_t if caching_helper else None,
                                    0 if caching_helper else None,
                                    None if seasons is None else seasons[0])
                        if t is not None))
      
    train_pred = np.array(real_pred_function(*(t for t in (model,
                                                           training_input,
                                                           target_t if caching_helper else None,
                                                           1 if caching_helper else None,
                                                           None if seasons is None else seasons[0])
                                               if t is not None)))
    test_pred = np.array(real_pred_function(*(t for t in (model,
                                                          validation_input,
                                                          target_t if caching_helper else None,
                                                          2 if caching_helper else None,
                                                          None if seasons is None else seasons[1])
                                              if t is not None)))
    
    train_res = np.array(training_output[target_column] - train_pred)
    train_MAE = mean_absolute_error(training_output[target_column], train_pred)
    test_res = np.array(validation_output[target_column] - test_pred)
    test_MAE = mean_absolute_error(validation_output[target_column], test_pred)
    
    if target_t == 24 and not skip_competition_predictions:
      real_fit_function(*(t for t in (model,
                                      competition_train_input,
                                      competition_train_output[target_column],
                                      target_t if caching_helper else None,
                                      3 if caching_helper else None,
                                      None if seasons is None else seasons[2])
                          if t is not None))
      competition_pred = np.array(real_pred_function(*(t for t in (model,
                                                                   competition_pred_input,
                                                                   target_t if caching_helper else None,
                                                                   4 if caching_helper else None,
                                                                   None if seasons is None else seasons[3])
                                                       if t is not None)))
      
      ret += [((train_pred, train_res, train_MAE), (test_pred, test_res, test_MAE), competition_pred)]
    else:
      ret += [((train_pred, train_res, train_MAE), (test_pred, test_res, test_MAE))]
  
  return ret

In [ ]:
#Function that prints a LaTeX table for a comparison of models.
#Receives an iterable whose contents are a dictionary
#that must contain at least "name", "train_MAE" and "val_MAE",
#with the latter two being a vector of as many numbers
#as the provided targets (optional argument that defaults to all).
#Optionally considering a factor for e. g. scaled output data.
#Can also specify an order in which to plot.
def print_model_comparison(models,
                           targets = (12, 24, 48),
                           factor = [1., 1., 1.],
                           model_order = None):
  #Print LaTeX table with the results
  table = "\\begin{tabular}{l | cc | cc | cc}\n\\hline\n\\textbf{Model}"
  for t in targets:
    table += "\n& \\multicolumn{2}{c|}{\\textbf{" + str(t) + " hours}}"
  table += " \\\\\n\\cline{2-" + str(2 * len(targets) + 1) + "}"
  for _ in targets:
    table += "\n& \\textbf{Train} & \\textbf{Test}"
  table += " \\\\\n\\cline{2-" + str(2 * len(targets) + 1) + "}"
  for m in models if model_order is None else  model_order:
    table += "\n" + models[m]["name"]
    for i in range(len(targets)):
      table += " & {:.3f} & {:.3f}".format(models[m]["train_MAE"][i] * factor[i], models[m]["val_MAE"][i] * factor[i])
  table += "\n\\hline\\end{tabular}"
  return table

In [ ]:
#Function that saves a plot of the model predictions.
#Receives an iterable whose contents are a dictionary
#that must contain at least "plot_label" and "val_pred",
#with the latter being a vector of 3 numbers (for the three targets),
#a dataframe containing the true values (for the three targets),
#and the name of the file where to save the plot.
#Optional parameters are:
#an iterable containing the target
#durations (if only a subset of 12, 24 and 48 is desired);
#whether to add labels (defaults to true);
#whether to actually save (defaults to true);
#whether to also plot the residuals (defaults to false);
#the colors to use for the models.
#Optionally also considering a factor and offset for e. g. scaled output data.
#Can also specify an order in which to plot.
def plot_model_comparison(models, true_values, file_name,
                          targets = (12, 24, 48),
                          add_label = True,
                          do_save = True,
                          also_residuals = False,
                          residuals_file_name = None,
                          colors = ['#0072B2', '#E69F00', '#009E73', '#D55E00'],
                          factor = 1.,
                          offset = 0.,
                          model_order = None):
  #First, combine into dataframe
  list_of_frames = []
 
  for m in models if model_order is None else model_order:
    this_frame = pd.DataFrame()
    this_frame["y_true"] = true_values[[f"target_tre200h0_plus{t}h" for t in targets]].to_numpy().flatten('F') * factor + offset
    this_frame["y_pred"] = np.concatenate(models[m]["val_pred"]) * factor + offset
    this_frame["y_res"] = this_frame["y_true"] - this_frame["y_pred"]
    this_frame["Model"] = models[m]["plot_label"]
    this_frame["time"] = np.concatenate([np.full(len(true_values), f"{t} hours") for t in targets])
    list_of_frames += [this_frame]
  
  plot_frame = pd.concat(list_of_frames)
  
  #Faceted scatter plot
  plot = (
      ggplot(plot_frame, aes(x='y_true', y='y_pred', color='Model'))
      + geom_point(alpha=0.7, size=1) 
      + geom_line(plot_frame, aes(x='y_true', y='y_true'), color='black')  # x=y line
      + facet_wrap('~time', scales='free')  # one facet per time point
      + labs(x='True Values', y='Predicted Values')
      + theme_bw()
      + scale_color_manual(values = colors)
  )
  if not add_label:
    plot += theme(legend_position="none")
  if do_save:
    plot.save(file_name, width=12, height=4, dpi=150)
  if also_residuals: 
  #Faceted scatter plot
    plot = (
        ggplot(plot_frame, aes(x='y_true', y='y_res', color='Model'))
        + geom_point(alpha=0.7, size=1) 
        + facet_wrap('~time', scales='free')  # one facet per time point
        + labs(x='True Values', y='Residuals')
        + theme_bw()
        + scale_color_manual(values = colors)
    )
    if not add_label:
      plot += theme(legend_position="none")
    if do_save:
      plot.save(residuals_file_name, width=12, height=4, dpi=150)
    
  return plot

## Station Map

In [ ]:
station_map_do_save = True
station_map_file_name = "switzerland_stations.png"

#Coordinates for the stations (latitude, longitude)
stations = {
    "Andermatt": (46.6309138888889, 8.58055277777778),
    "Basel": (47.5411416666667, 7.583525), 
    "Davos": (46.8129694444444, 9.84355833333333),
    "La Dôle": (46.4247944444444, 6.09945277777778),
    "Genève": (46.1957416666667, 6.09050833333333),
    "Interlaken": (46.6722333333333, 7.87019444444444),
    "Lugano": (46.0042166666667, 8.96032222222222),
    "Sion": (46.21865, 7.33020277777778),
    "St-Gallen": (47.425475, 9.39852777777778),
    "Zermatt": (46.0292722222222, 7.75243333333333)
}

bern = (46.9907444444444, 7.46406111111111)

#Create a figure with Cartopy projection
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([5.75, 10.75, 45.5, 48], crs=ccrs.PlateCarree()) 
#Add land, borders
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.BORDERS, linestyle=':')

#Stations
for name, (lat, lon) in stations.items():
    ax.plot(lon, lat, 'ro', markersize=5, transform=ccrs.PlateCarree())
    ax.text(lon + 0.025, lat + 0.025, name, transform=ccrs.PlateCarree())
#Bern
ax.plot(bern[1], bern[0], marker='*', color='blue', markersize=12, transform=ccrs.PlateCarree())
ax.text(bern[1] + 0.05, bern[0] + 0.05, 'Bern', fontsize=10, fontweight='bold', transform=ccrs.PlateCarree())


if station_map_do_save:
  plt.savefig(station_map_file_name, dpi=150, bbox_inches='tight')

plt.show()

## Training and Validation Data Splitting and Scaling

In [ ]:
input_data_file = "train.csv"
competition_data_file = "TEST.csv"
training_validation_split_test_fraction = 0.2
training_validation_split_random_seed = 42

full_X, full_Y, full_seasons = load_data(input_data_file)
competition_X, _, competition_seasons = load_data(competition_data_file) #Kaggle

X_train, X_test, Y_train, Y_test, seasons_train, seasons_test = train_test_split(full_X, full_Y, full_seasons,
                                                                                 shuffle=True,
                                                                                 test_size = training_validation_split_test_fraction,
                                                                                 random_state = training_validation_split_random_seed)

#We scale all input that will be used for training and validation
#according to X_train.
#Kaggle predictions will use the full data set
#for scaling (and training).
overall_scaling     = get_data_transformers(X_train)
competition_scaling = get_data_transformers(full_X)

#Also determine the overall Y scaling
#if we want to pass predictions into the second level
#of the two-tier model
overall_Y_scaler = StandardScaler().fit(Y_train)
overall_Y_offset = np.average(overall_Y_scaler.mean_)
overall_Y_scale = np.sqrt(np.average(overall_Y_scaler.var_))

X_train_scaled = transform_data(overall_scaling, X_train)
X_test_scaled  = transform_data(overall_scaling, X_test)

full_X_scaled        = transform_data(competition_scaling, full_X)
competition_X_scaled = transform_data(competition_scaling, competition_X)


## Descriptive Overview

In [ ]:
#Temperature histograms
overview_temperature_do_save = True
overview_temperature_file_name = "temperature_hist.png"

#Reshape data into long format for faceting
frame_for_plotting = full_Y.melt(
    value_vars=["target_tre200h0_plus12h", "target_tre200h0_plus24h", "target_tre200h0_plus48h"],
    var_name="target_hour", value_name="temperature"
)

#Map variable names to the desired labels
label_map = {
    "target_tre200h0_plus12h": "12 hours",
    "target_tre200h0_plus24h": "24 hours",
    "target_tre200h0_plus48h": "48 hours"
}
frame_for_plotting["target_hour"] = frame_for_plotting["target_hour"].map(label_map)

#Build ggplot with facets
plot = (ggplot(frame_for_plotting, aes(x="temperature"))
     + geom_histogram(bins=20, fill="gray", color="black")
     + facet_wrap("~target_hour", scales="free")
     + labs(x="Temperature", y="Frequency")
     + theme_bw() + theme(figure_size=(12, 4))
    )

if overview_temperature_do_save:
  plot.save(overview_temperature_file_name, dpi=150)

plot

In [ ]:
#Temperature distribution per season and by hour
overview_season_do_save = True
overview_season_file_name = "season_dist.png"
overview_hour_do_save = True
overview_hour_file_name = "hour_dist.png"

frame_for_plotting = full_X[["season_sin", "season_cos", "hour_sin", "hour_cos", "tre200h0"]].copy()

frame_for_plotting["season"] = "UNKNOWN"
frame_for_plotting["hour"] = "UNKNOWN"

seasons = ('Spring', 'Summer', 'Autumn', 'Winter')
hours = [str(i) for i in range(24)]

#Regenerate season from sine and cosine since we dropped it
for j, label in enumerate(seasons):
  frame_for_plotting.loc[(frame_for_plotting['season_sin'] == np.sin(2 * np.pi * j / 4)) * (frame_for_plotting['season_cos'] == np.cos(2 * np.pi * j / 4)) , "season"] = label

#Same for hour
for j, label in enumerate(hours):
  frame_for_plotting.loc[(frame_for_plotting['hour_sin'] == np.sin(2 * np.pi * j / 24)) * (frame_for_plotting['hour_cos'] == np.cos(2 * np.pi * j / 24)) , "hour"] = label

#Ensure right order for ggplot
frame_for_plotting["season"] = pd.Categorical(frame_for_plotting["season"], categories = seasons, ordered = True)

plot1 = (ggplot(frame_for_plotting, aes(x="season", y="tre200h0"))
     + geom_boxplot(fill="gray")
     + labs(x="Season", y="Temperature")
     + theme_bw()
     + theme(figure_size=(8, 5))
    )

plot2 = (
    ggplot(frame_for_plotting, aes(x="hour", y="tre200h0"))
    + geom_boxplot(fill="gray")
    + scale_x_discrete(limits=hours)
    + facet_wrap("~season", ncol=2)     
    + labs(x="Hour", y="Temperature")
    + theme_bw()
    + theme(figure_size=(12, 8))
)

if overview_season_do_save:
  plot1.save(overview_season_file_name, dpi=150)
    
if overview_hour_do_save:
  plot2.save(overview_hour_file_name, dpi=150)

In [ ]:
plot1

In [ ]:
plot2

In [ ]:
#Heatmap
overview_heatmap_do_save = True
overview_heatmap_file_name = "corr_heatmap.png"

frame_for_plotting = pd.concat([full_X, full_Y], axis=1)

corr_matrix = frame_for_plotting.corr()

plt.figure(figsize=(22, 22))
sns.heatmap(corr_matrix, cmap='coolwarm')

if overview_heatmap_do_save:
  plt.savefig(overview_heatmap_file_name, dpi=100, bbox_inches='tight') 

plt.show()

In [ ]:
#Distribution of global radiation
radiation_do_save = True
radiation_file_name = "radiation_hist.png"

plot = (ggplot(full_X, aes(x="gre000h0_BAS"))
     + geom_histogram(bins=20, fill="gray", color="black")
     + labs(x= "Global Radiation", y="Frequency") +theme_bw())

if radiation_do_save:
  plot.save(radiation_file_name, dpi=150)

plot

In [ ]:
#Distribution of precipitation
precipitation_do_save = True
precipitation_file_name = "precipitation_hist.png"

plot = (ggplot(full_X, aes(x="rre150h0_BAS"))
     + geom_histogram(bins=20, fill="gray", color="black")
     + labs(x= "Precipitation", y="Frequency") +theme_bw())

if precipitation_do_save:
  plot.save(precipitation_file_name, dpi=150)

plot

In [ ]:
#Distribution of sunshine duration
sunshine_do_save = True
sunshine_file_name = "sunshine_hist.png"

plot = (ggplot(full_X, aes(x="sre000h0_BAS"))
     + geom_histogram(bins=20, fill="gray", color="black")
     + labs(x= "Sunshine Duration", y="Frequency") +theme_bw())

if sunshine_do_save:
  plot.save(sunshine_file_name, dpi=150)

plot

In [ ]:
#Distribution of average wind speed
wind_do_save = True
wind_file_name = "wind_hist.png"

plot = (ggplot(full_X, aes(x="fkl010h0_BAS"))
     + geom_histogram(bins=20, fill="gray", color="black")
     + labs(x= "Wind Speed", y="Frequency") +theme_bw())

if wind_do_save:
  plot.save(wind_file_name, dpi=150)

plot

## Basic Models

### Defining the Models

In [ ]:
basic_random_forest_random_seed = 42
basic_boosting_random_seed = 42

basic_num_cores_to_consider = -1 


linear_regression_model = { "name": "Linear Regression",
                            "plot_label": "Linear Reg.",
                            "create_model": (lambda p: LinearRegression()),
                            "get_params": (lambda t: None),
                            "do_tune": False, #No hyperparameters to tune
                            "tuning_n_jobs": 1 #Irrelevant here
                          }

random_forest_model = { "name": "Random Forest",
                        "plot_label": "Random Forest",
                        "create_model": (lambda p: RandomForestRegressor(**p)),
                        "get_params": (lambda t: {
                          'n_estimators': t.suggest_int('n_estimators', 100, 1000, log=True),
                          'max_depth': t.suggest_int('max_depth', 10, 50),
                          'min_samples_leaf': t.suggest_int('min_samples_leaf', 1, 10),
                          'max_features': t.suggest_float('max_features', 0.1, 1.0),
                          'n_jobs': basic_num_cores_to_consider,
                          'random_state': basic_random_forest_random_seed
                        }),
                        "do_tune": True,
                        "tuning_n_jobs": 1 #Fitting each forest is already parallelized
                      }

SVR_model = { "name": "SVR",
              "plot_label": "SVR",
              "create_model": (lambda p: SVR(kernel="rbf", gamma="scale", **p)),
              "get_params": (lambda t: {
                'C': t.suggest_float("C", 2.5, 50, log=True),
                'epsilon': t.suggest_float("epsilon", 0.1, 1.0, log=True)
              }),
              "do_tune": True,
              "tuning_n_jobs": basic_num_cores_to_consider #Since SVR fitting is not parallelized
            }

def boosting_CV(trial, params, model, folds, data_X, data_Y):
  d_matrix = xgboost.DMatrix(data_X, label = data_Y)
  res = xgboost.cv(params = params,
                   dtrain = d_matrix,
                   num_boost_round = 2500,
                   early_stopping_rounds = 25,
                   seed = basic_boosting_random_seed,
                   metrics = ['mae'],
                   maximize = False,
                   folds = folds,
                   show_stdv = False,
                   verbose_eval = False,
                   callbacks = [optuna_integration.XGBoostPruningCallback(trial, "test-mae")])
  trial.set_user_attr("n_estimators", len(res))
  return res["test-mae-mean"].values[-1]


boosting_model = { "name": "Boosting",
                   "plot_label": "XGBoost",
                   "create_model": (lambda p: xgboost.XGBRegressor(**p)),
                   "get_params": (lambda t: {
                     'learning_rate': t.suggest_float('learning_rate', 0.001, 0.1, log=True),
                     'max_depth': t.suggest_int('max_depth', 3, 10),
                     'subsample': t.suggest_float('subsample', 0.5, 1.0),
                     'colsample_bytree': t.suggest_float('colsample_bytree', 0.5, 1.0),
                     'min_child_weight': t.suggest_float('min_child_weight', 1, 10),
                     'reg_lambda': t.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
                     'reg_alpha': t.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                     'gamma': t.suggest_float('gamma', 1e-5, 10.0, log=True),
                     'objective': "reg:squarederror",
                     'eval_metric': "mae",
                     'n_jobs': basic_num_cores_to_consider,
                     'random_state': basic_boosting_random_seed
                   }),
                   "override_CV": boosting_CV,
                   "do_tune": True,
                   "tuning_n_jobs": 1 #Boosting is already parallelized
                 }

basic_models = {'linear_regression': linear_regression_model, 'forest': random_forest_model, 'SVR': SVR_model, 'boosting': boosting_model}

### Tuning the Models

In [ ]:
for m in basic_models:
  if not basic_models[m]["do_tune"]:
    basic_models[m]["tuning_MAE"] = -1
    basic_models[m]["best_params"] = [None, None, None]
    continue
  print("Tuning: ", basic_models[m]["name"])
  
  tuning_result = tune_model(basic_models[m]["create_model"],
                             basic_models[m]["get_params"],
                             X_train_scaled,
                             Y_train,
                             tuning_n_jobs = basic_models[m]["tuning_n_jobs"],
                             num_total_trials = 100,
                             override_CV = basic_models[m]["override_CV"] if "override_CV" in basic_models[m] else None)
  
  basic_models[m]["tuning_MAE"]  = [mae for mae, _ in tuning_result]
  basic_models[m]["best_params"] = [params for _, params in tuning_result]

  print("Finished: ", basic_models[m]["name"])
print("Done!")

### Save Best Parameters to File

In [ ]:
basic_models_file_name = "basic_models.json"

output_dictionary = {}

for m in basic_models:
  output_dictionary[m] = {k: v for k, v in basic_models[m].items() if k in ('tuning_MAE', 'best_params')}

with open(basic_models_file_name, "w") as output_file:
  json.dump(output_dictionary, output_file, indent=1)
print(output_dictionary)

### Reload Best Parameters from File (if needed)

In [ ]:
basic_models_file_name = "basic_models.json"

input_dictionary = {}

with open(basic_models_file_name, "r") as input_file:
  input_dictionary = json.load(input_file)

for k, v in input_dictionary.items():
  basic_models[k].update(v)

### Assessing the Models

In [ ]:
for m in basic_models:
  print("Assessing {}".format(basic_models[m]["name"]))
  results = assess_model( basic_models[m]["create_model"], basic_models[m]["best_params"],
                          X_train_scaled, Y_train, X_test_scaled, Y_test,
                          full_X_scaled, full_Y, competition_X_scaled )
  (basic_models[m]["train_pred"], basic_models[m]["train_res"], basic_models[m]["train_MAE"],
   basic_models[m]["val_pred"],   basic_models[m]["val_res"],   basic_models[m]["val_MAE"]    ) = ([t[i][j] for t in results] for i in range(0, 2) for j in range(0, 3))
  basic_models[m]["competition_pred"] = results[1][2]
  
  print("Finished {}".format(basic_models[m]["name"]))
print("Done!")

In [ ]:
#Print LaTeX table with the results
table_code = print_model_comparison(basic_models, model_order = ('linear_regression', 'forest', 'SVR', 'boosting'))
print(table_code)

In [ ]:
#Plot using ggplot
plot_model_comparison(basic_models, Y_test,
                      "predictions_facets.png",
                      model_order = ('linear_regression', 'forest', 'boosting', 'SVR'))

## Sophisticated Model

### Generic Definitions

In [ ]:
default_sophisticated_model_n_folds_tuning  = 7 #k
default_sophisticated_model_n_folds_fitting = 20 #m

default_sophisticated_model_random_seed = 42

In [ ]:
#Tune the best hyperparameters for the second model,
#assuming the first's hyperparameters are "good enough".
#Returns the pre-calculated things to save time if the next
#models to fit have the same first level.
#level_one_model and level_two_model are dictionaries
#like the ones used in the basic model section,
#containing at least "name, "create_model" and "get_params",
#for level_one_model, also "best_params", 
#for level_two_model, also "tuning_n_jobs".
#Optional arguments:
#a function that can modify the input to the second model
#(which receives (a copy of) the input data
# for the relevant range (e.g. training or tuning folds)
# and the output of the first model to be used there);
#whether to train the second level to output the final values
#instead of the residuals;
#the number of folds to use for tuning the second model;
#the number of folds to use for training the first model and calculating OOFs;
#an iterable containing the target
#durations (if only a subset of 12, 24 and 48 is desired);
#the total number of trials to tune (defaulting to 100);
#whether to show the optimisation plot for each target (defaulting to False);
#cached precalculated results of the first level to skip redundant work
#when testing multiple models with the same first level
def sophisticated_models_tune(level_one_model,
                              level_two_model,
                              training_input,
                              training_output,
                              training_seasons,
                              second_level_input = None,
                              just_second_level_output = False,
                              n_folds_tuning = default_sophisticated_model_n_folds_tuning,
                              n_folds_fitting = default_sophisticated_model_n_folds_fitting,
                              targets = (12, 24, 48),
                              num_tuning_trials = 100,
                              show_optimisation_plot = False,
                              previous_results = None):
  if previous_results is None:
    tuning_folder  = KFold(n_splits = n_folds_tuning,
                           random_state = default_sophisticated_model_random_seed,
                           shuffle = True)
    fitting_folder = KFold(n_splits = n_folds_fitting,
                           random_state = default_sophisticated_model_random_seed,
                           shuffle = True)
    tuning_folds = []
    #Precalculate OOF residuals
    predictions_train = {}
    predictions_test = {}
    OOF_residuals_train =  {}
    OOF_residuals_test  = {}
    for column_idx, target in enumerate(targets):
      target_column = "target_tre200h0_plus{}h".format(target)
      predictions_train[target_column] = []
      predictions_test[target_column] = []
      OOF_residuals_train[target_column] = []
      OOF_residuals_test[target_column] = []
      this_train_output = training_output[target_column]
      
      #We will take this as fixed for building the sophisticated model.
      #The underlying assumption is that the best individually tuned parameters
      #are likely to be "good enough" for the combined model.
      #If this was allowed to change, the tuning procedure
      #would take much more time as we would need to fit the first model
      #and calculate OOFs at every iteration...
      first_level = level_one_model["create_model"](level_one_model["best_params"][column_idx])
      
      #Tuning folder should return a pair of vectors
      #containing the indices to use for training and testing, respectively
      for i, (folds_for_training, fold_for_testing) in enumerate(tuning_folder.split(training_input)):
        if (column_idx == 0):
          tuning_folds += [(folds_for_training, fold_for_testing)]
    
        print(f"Pre-calculating {target}h: tuning {i}")
        predictions = np.zeros(len(folds_for_training))
        
        #Calculate the OOF predictions by training the first model
        #on the n_folds_fitting - 1 folds and predicting
        #on the unseen fold
        for j, (in_fold, out_of_fold) in enumerate(fitting_folder.split(folds_for_training)):
          this_training_indices = folds_for_training[in_fold]
          this_residual_indices = folds_for_training[out_of_fold]
          print(f"                     tuning {i} testing {j}")
          first_level.fit(training_input.iloc[this_training_indices], this_train_output.iloc[this_training_indices])
          predictions[out_of_fold] = np.array(first_level.predict(training_input.iloc[this_residual_indices]))
        
        predictions_train[target_column] += [predictions]
        OOF_residuals_train[target_column] += [np.array(this_train_output.iloc[folds_for_training] - predictions)]
        
        #And now the model trained over all data
        #for getting the residuals in the testing fold
        first_level.fit(training_input.iloc[folds_for_training], this_train_output.iloc[folds_for_training])
        
        test_predictions = first_level.predict(training_input.iloc[fold_for_testing])
        
        predictions_test[target_column] += [test_predictions]
        OOF_residuals_test[target_column] += [np.array(this_train_output.iloc[fold_for_testing] - test_predictions)]
    
    previous_results = (tuning_folds, predictions_train, OOF_residuals_train, predictions_test, OOF_residuals_test)
  else:
    tuning_folds, predictions_train, OOF_residuals_train, predictions_test, OOF_residuals_test = previous_results
    
  if second_level_input is not None:
    real_tuning_input = {column: [( second_level_input(training_input.iloc[train_fs].copy(), train_pred),
                                    second_level_input(training_input.iloc[test_fs].copy(), test_pred)    )
                                  for (train_fs, test_fs), train_pred, test_pred in zip(tuning_folds,
                                                                                        predictions_train[column],
                                                                                        predictions_test[column])]
                         for column in OOF_residuals_test}
    tuning_input = lambda column, idx, t_idx: real_tuning_input[column][idx][0]
    validation_input = lambda column, idx, t_idx: real_tuning_input[column][idx][1]
  else:
    tuning_input = training_input
    validation_input = None

  if just_second_level_output:
    tuning_output = training_output
    validation_output = None
  else:
    tuning_output = lambda column, idx, t_idx: OOF_residuals_train[column][idx]
    validation_output = lambda column, idx, v_idx: OOF_residuals_test[column][idx]

  return previous_results, tune_model(level_two_model["create_model"],
                                      level_two_model["get_params"],
                                      tuning_input,
                                      tuning_output,
                                      validation_input = validation_input,
                                      validation_output = validation_output,
                                      split_folds = (lambda x: tuning_folds),
                                      #We want to unconditionally return the exact folds for which we pre-calculated.
                                      #Could just use tuning_folder.split since it is deterministic,
                                      #but this way we are super-sure!
                                      num_total_trials = num_tuning_trials,
                                      show_optimisation_plot = show_optimisation_plot,
                                      tuning_n_jobs = level_two_model["tuning_n_jobs"],
                                      seasons = training_seasons)


In [ ]:
#Fit actual instances of a model,
#possibly with a custom input to the second stage.
#If seasons is not None, we use one model per season
#and expect second_level to be an adequately sized list.
def sophisticated_models_fit(first_level,
                             second_level,
                             fit_X,
                             fit_Y,
                             seasons = None,
                             second_level_input = None,
                             just_second_level_output = False,
                             num_folds = default_sophisticated_model_n_folds_fitting,
                             caching_helper = None):
  if caching_helper is None or len(caching_helper) == 0:
    predictions = np.zeros(len(fit_X))
    fit_folds = KFold(n_splits = num_folds,
                      random_state = default_sophisticated_model_random_seed,
                      shuffle = True)
  
    #Calculate OOF predictions with the first model
    for i, (train_fold, test_fold) in enumerate(fit_folds.split(fit_X)):
      print(i)
      first_level.fit(fit_X.iloc[train_fold], fit_Y.iloc[train_fold])
      predictions[test_fold] = np.array(first_level.predict(fit_X.iloc[test_fold]))
        
    #Train first level model over all input data
    #only if we haven't cached values yet
    #(in prediction)
    first_level.fit(fit_X, fit_Y)
    
    if caching_helper is not None:
      caching_helper += [predictions]
  else:
    predictions = caching_helper[0]

  second_input = fit_X if second_level_input is None else second_level_input(fit_X.copy(), predictions)

  second_output = fit_Y if just_second_level_output else fit_Y - predictions

  if seasons is not None:
    #Train a second level model over all OOF residuals
    #for every season.
    for i, u in enumerate(seasons.unique()):
      locator = np.array(seasons == u)
      second_level[i].fit(second_input[locator], second_output[locator])
  else:
    second_level.fit(second_input, second_output)

  
  return (first_level, second_level)


In [ ]:
#Obtain the predictions for (scaled) data,
#possibly with a custom input to the second stage.
#If seasons is not None, we use one model per season
#and expect second_level to be an adequately sized list.
def sophisticated_models_predict(first_level,
                                 second_level,
                                 pred_X,
                                 seasons = None,
                                 second_level_input = None,
                                 just_second_level_output = False,
                                 caching_helper = None):
  if caching_helper is None or len(caching_helper) == 0:
    first_level_pred = first_level.predict(pred_X)
    if caching_helper is not None:
      caching_helper += [first_level_pred]
  else:
    first_level_pred = caching_helper[0]
    
  second_input = pred_X if second_level_input is None else second_level_input(pred_X.copy(), first_level_pred)

  second_level_pred = np.zeros(len(first_level_pred))
  
  if seasons is not None:
    for i, u in enumerate(seasons.unique()):
      locator = np.array(seasons == u)
      second_level_pred[locator] = second_level[i].predict(second_input[locator])
  else:
    second_level_pred = second_level.predict(second_input)

  if just_second_level_output:
    return second_level_pred
  else:
    return first_level_pred + second_level_pred

### Defining the Models

In [ ]:
def add_pred_to_second_level(X_copy, preds):
  X_copy["preds"] = (preds - overall_Y_offset)/overall_Y_scale
  return X_copy

boosting_svr_standard = {"name": "Boosting + SVR",
                         "plot_label": "Boosting + SVR",
                         "first": basic_models['boosting'],
                         "second": basic_models['SVR'],
                         "second_level_input": None,
                         "just_second_level_output": False,
                         "do_per_season": False,
                         "precalc_type": 'boosting',
                         "do_tune": True }

boosting_svr_with_pred = {"name": "Boosting ⊕ SVR",
                          "plot_label": "Boosting ⊕ SVR",
                          "first": basic_models['boosting'],
                          "second": basic_models['SVR'],
                          "second_level_input": add_pred_to_second_level,
                          "just_second_level_output": False,
                          "do_per_season": False,
                          "precalc_type": 'boosting',
                          "do_tune": True }

boosting_svr_with_pred_direct = {"name": "Boosting → SVR",
                                 "plot_label": "Boosting → SVR",
                                 "first": basic_models['boosting'],
                                 "second": basic_models['SVR'],
                                 "second_level_input": add_pred_to_second_level,
                                 "just_second_level_output": True,
                                 "do_per_season": False,
                                 "precalc_type": 'boosting',
                                 "do_tune": True }

sophisticated_models = {'bs1': boosting_svr_standard,
                        'bs2': boosting_svr_with_pred,
                        'bs3': boosting_svr_with_pred_direct
                       }

### Tuning the Sophisticated Models

In [ ]:
precalculated_cache = {}

In [ ]:
for m in sophisticated_models:
  if not sophisticated_models[m]["do_tune"]:
    sophisticated_models[m]["tuning_MAE"] = -1
    sophisticated_models[m]["best_params"] = [None, None, None]
    continue
  print("Tuning: ", sophisticated_models[m]["name"])

  this_model_cache = sophisticated_models[m]["precalc_type"]

  if this_model_cache not in precalculated_cache:
    prev_values = None
  else:
    prev_values = precalculated_cache[this_model_cache]
  
  prev_values, tuning_result = sophisticated_models_tune(sophisticated_models[m]["first"],
                                                         sophisticated_models[m]["second"],
                                                         X_train_scaled,
                                                         Y_train,
                                                         seasons_train if sophisticated_models[m]["do_per_season"] else None,
                                                         second_level_input = sophisticated_models[m]["second_level_input"],
                                                         just_second_level_output = sophisticated_models[m]["just_second_level_output"],
                                                         num_tuning_trials = 100,
                                                         previous_results = prev_values)
  
  if this_model_cache not in precalculated_cache:
    precalculated_cache[this_model_cache] = prev_values
  
  sophisticated_models[m]["tuning_MAE"]  = [mae for mae, _ in tuning_result]
  sophisticated_models[m]["best_params"] = [params for _, params in tuning_result]
  
  print("Finished: ", sophisticated_models[m]["name"])
print("Done!")


### Save Best Parameters to File

In [ ]:
sophisticated_models_file_name = "sophisticated_models.json"

output_dictionary = {}

for m in sophisticated_models:
  output_dictionary[m] = {k: v for k, v in sophisticated_models[m].items() if k in ('tuning_MAE', 'best_params')}

with open(sophisticated_models_file_name, "w") as output_file:
  json.dump(output_dictionary, output_file, indent=1)
print(output_dictionary)

### Reload Best Parameters from File (if needed)

In [ ]:
sophisticated_models_file_name = "sophisticated_models.json"

input_dictionary = {}

with open(sophisticated_models_file_name, "r") as input_file:
  input_dictionary = json.load(input_file)

for k, v in sophisticated_models.items():
  sophisticated_models[k].update(v)

### Assessing the Sophisticated Models

In [ ]:
assessment_cache = {}

In [ ]:
for m in sophisticated_models:
  print("Assessing {}".format(sophisticated_models[m]["name"]))
  
  this_model_cache = sophisticated_models[m]["precalc_type"]
    
  if this_model_cache not in assessment_cache:
    assessment_cache[this_model_cache] = {}

  if sophisticated_models[m]["do_per_season"]:
    model_maker = lambda p: ( sophisticated_models[m]["first"]["create_model"](p[0]),
                              [sophisticated_models[m]["second"]["create_model"](p[1]) for _ in full_seasons.unique()] )
  else:
    model_maker = lambda p: ( sophisticated_models[m]["first"]["create_model"](p[0]),
                              sophisticated_models[m]["second"]["create_model"](p[1]) )
    
  model_params = [z for z in zip(sophisticated_models[m]["first"]["best_params"],
                                 sophisticated_models[m]["best_params"])
                 ]
  
  model_fitter = lambda model, x, y, t, i, s = None: sophisticated_models_fit(model[0],
                                                                              model[1], 
                                                                              x,
                                                                              y,
                                                                              seasons = s,
                                                                              second_level_input = sophisticated_models[m]["second_level_input"],
                                                                              just_second_level_output = sophisticated_models[m]["just_second_level_output"],
                                                                              caching_helper = assessment_cache[this_model_cache].setdefault(t,[[] for _ in range(5)])[i])
  
  model_predictor = lambda model, x, t, i, s = None: sophisticated_models_predict(model[0],
                                                                                  model[1], 
                                                                                  x,
                                                                                  seasons = s,
                                                                                  second_level_input = sophisticated_models[m]["second_level_input"],
                                                                                  just_second_level_output = sophisticated_models[m]["just_second_level_output"],
                                                                                  caching_helper = assessment_cache[this_model_cache].setdefault(t,[[] for _ in range(5)])[i])
  

  results = assess_model( model_maker, model_params,
                          X_train_scaled, Y_train, X_test_scaled, Y_test,
                          full_X_scaled, full_Y, competition_X_scaled,
                          fit_model = model_fitter,
                          predict_with_model = model_predictor,
                          caching_helper = True,
                          seasons = None if not sophisticated_models[m]["do_per_season"] else (seasons_train, seasons_test, full_seasons, competition_seasons))
  ( sophisticated_models[m]["train_pred"],
    sophisticated_models[m]["train_res"],
    sophisticated_models[m]["train_MAE"],
    sophisticated_models[m]["val_pred"],
    sophisticated_models[m]["val_res"],
    sophisticated_models[m]["val_MAE"]      ) = ([t[i][j] for t in results] for i in range(0, 2) for j in range(0, 3))
  sophisticated_models[m]["competition_pred"] = results[1][2]
  
  print("Finished {}".format(sophisticated_models[m]["name"]))
print("Done!")
  

In [ ]:
#Print LaTeX table with the results
table_code = print_model_comparison(sophisticated_models)
print(table_code)

#Plot using ggplot
plot_model_comparison(sophisticated_models, Y_test,
                      "predictions_facets_sophis.png", model_order = ('bs1', 'bs2', 'bs3'))

## Output for Kaggle

In [ ]:
kaggle_submission_file = "kaggle_submission_19.csv"

best_model = sophisticated_models["bs2"]

submission = pd.DataFrame({"target_tre200h0_plus24h" : best_model["competition_pred"]})

submission["Id"] = submission.index + 1

#Note: we must specify the lineterminator to be '\n'
#because of the end-of-line encoding differences
#between different operating systems.
#When writing to the output file below,
#'\n' is encoded to the right thing based on the OS.
to_write = submission.to_csv(None, index=False, lineterminator='\n', columns=("Id", "target_tre200h0_plus24h")).replace(',', ', ')

with open(kaggle_submission_file, "w") as output_file:
  output_file.write(to_write)